In [ ]:
from stop_words import get_stop_words

In [ ]:
!pip install nltk

In [ ]:
from nltk.corpus import stopwords

# Import Libraries 

In [ ]:
import pandas, numpy
from bs4 import BeautifulSoup
import re, os, glob, sklearn
from sklearn.linear_model import LogisticRegression
#import stopwords
import nltk
from nltk.corpus import stopwords
import numpy as np

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:

dataset=pandas.read_csv('/kaggle/input/pakistani-traffic-sentiment-analysis/Pakistani Traffic sentiment Analysis.csv')
print (dataset)
text=dataset.loc[:, 'Text']
print (text)
label=dataset.loc[:,	'Label']
print (label)

In [ ]:
def clean(data):
	review_text=BeautifulSoup(data, "html").get_text()
	letters_only=re.sub("[^a-zA-Z]", "", review_text)
	words=letters_only.lower().split()
	stop_words=set(stopwords.words('english'))
	meaningful_words=[w for w in words if not w in stop_words]
	return("".join(meaningful_words))
clean_train_reviews=[]
for i in range(0, len(text)):
	print (i), len(text)
	clean_train_reviews.append(clean(text[i]))
print (clean_train_reviews)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
tfidfconverter = CountVectorizer()
X = tfidfconverter.fit_transform(clean_train_reviews).toarray()
y = dataset.iloc[:, 1]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
classifier = LogisticRegression()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
classifier = LogisticRegression()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [ ]:
print(confusion_matrix(y_test,y_pred))


In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
print(accuracy_score(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# roc curve and auc score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
def plot_roc_curve(fpr, tpr):
    plt.plot(fpr, tpr, color='orange', label='ROC')
    plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend()
    plt.show()

In [ ]:
data_X, class_label = make_classification(n_samples=1000, n_classes=2, weights=[1,1], random_state=1)
trainX, testX, trainy, testy = train_test_split(data_X, class_label, test_size=0.2, random_state=1)
model = LogisticRegression()
model.fit(trainX, trainy)
probs = model.predict_proba(testX)
probs = probs[:, 1]
auc = roc_auc_score(testy, probs)
print('AUC: %.2f' % auc)
fpr, tpr, thresholds = roc_curve(testy, probs)
plot_roc_curve(fpr, tpr)

In [ ]:
fpr, tpr, thresholds = roc_curve(testy, probs)
plot_roc_curve(fpr, tpr)
# Plot non-normalized confusion matrix
def plot_confusion_matrix(y_true, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    # Only use the labels that appear in the data
    classes = classes[unique_labels(y_true, y_pred)]
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')
        print(cm)
        fig ,ax = plt.subplots()
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    # We want to show all ticks...

    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           # ... and label them with the respective list entries
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')

    # Rotate the tick labels and set their alignment.
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    # Loop over data dimensions and create text annotations.
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax